# Skin disease classification, step by step

We will clean the data, choose 20 diagnoses, split the images, and train ResNet18 or Swin-Tiny.
Pandas handles the tables. Scikit-learn handles splitting and scores. PyTorch trains
the neural network. Run the cells from top to bottom.

Use the local Derm1M CSVs and extracted image folders beside the notebook, or set
`download_dataset_from_hugging_face = True` in the settings to fetch both CSVs and
all image archives. See section 3 for setup and storage details.

The official CSV has no diagnosis labels, so we can export its predictions but
cannot calculate its accuracy. Our validation scores are internal experiment scores.


## 1. Import the libraries

These are the tools used below. There are no external Python helper files.


In [1]:
import io
import re
import time
import random
import hashlib

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps, ImageCms
from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights, swin_t, Swin_T_Weights

## Optional: log in to Hugging Face

Set `login_to_hugging_face=True` below and run the cell to sign in. Create a
[Hugging Face access token](https://huggingface.co/settings/tokens) and paste it
into the hidden input when prompted; do not paste it into the cell's code.
A read token is sufficient for downloads; uploading models requires write access.

This uses `huggingface_hub`, already included in `requirements.txt`. The library
caches your login in the current machine or runtime's Hugging Face cache, rather
than saving the token in the notebook or adding it to Git credentials.
The download cell uses your cached Hugging Face login. For Derm1M, first sign in
and accept the access conditions on the
[dataset page](https://huggingface.co/datasets/redlessone/Derm1M). Then use this
login cell if the runtime does not already have a read token.
See the [authentication guide](https://huggingface.co/docs/huggingface_hub/main/package_reference/authentication).


In [2]:
login_to_hugging_face = False

if login_to_hugging_face:
    from getpass import getpass
    from huggingface_hub import login

    hf_token = getpass("Hugging Face access token (hidden; leave blank to skip): ").strip()
    try:
        if hf_token:
            login(token=hf_token, add_to_git_credential=False)
        else:
            print("Hugging Face login skipped.")
    finally:
        del hf_token


## 2. Choose the experiment

Start with ImageNet-pretrained ResNet18. The saved ResNet18 runs provide a
comparison on the same validation split. Swin-Tiny is also available by
setting `model_name = "swin_t"`. Each architecture has its own run name.

By default, each training image is used at most once per epoch. Each source
group contributes up to eight images per diagnosis per epoch, and the
selected frames rotate. Square-root class weights give smaller diagnoses
more influence, and label smoothing reduces the incentive to memorize
individual labels. Set `images_per_group = None` to disable the cap.
`balanced_sampling = True` enables the earlier replacement sampling mode.

`split_reference_run = "dropout_040_run"` reuses the saved training and
validation manifests. Keep both CSVs beside the notebook. Set the reference
to an empty string when intentionally creating a different split. The
reference run's F1-selected checkpoint reached 51.22% validation accuracy
and 0.4186 macro F1.

Keep `fast_mode = False` for a full comparison. Fast mode limits training
examples per class and uses a smaller validation sample for checkpoint selection.
`reuse_image_checks = True` reuses the existing image audit. Set it to False
if the audit is unavailable or the source files changed.

Denoising is optional. In a four-epoch CPU comparison with 480 training images
per epoch and a fixed 240-image validation sample, resize-only scored 18.75%
accuracy and 0.145 macro F1. With bilateral denoising, it scored 16.67% and 0.127.
Resize-only is the default based on this pilot. A full run is needed to compare
the new settings with the saved baseline.

Set `denoise_images = True` to repeat the experiment with filtering. This changes
the default run name. Keep the split and seed fixed when comparing the two.
Use a fresh run name for other setting changes.


In [ ]:
model_name = "resnet18"  # Choose "resnet18" or "swin_t".
balanced_sampling = True
denoise_images = True
run_name = (
    f"{model_name}_{'balanced' if balanced_sampling else 'weighted'}"
    f"_{'denoised' if denoise_images else 'resize_only'}_v1"
)
split_reference_run = "dropout_040_run"  # Reuse this experiment's saved split.
resume_training = False  # Set True to continue from this run's best saved checkpoint.
download_dataset_from_hugging_face = False  # Download both CSVs and all image archives.
download_csvs_from_hugging_face = False  # Fetch just the CSVs when images are already local.
hf_dataset_directory = "../dataset/Derm1M"  # Where downloaded images are extracted.
hf_dataset_repo_id = "redlessone/Derm1M"
hf_dataset_revision = "main"  # Use a commit ID to pin the dataset version.
fast_mode = False
images_per_class = 300
validation_images_per_class = 50
reuse_image_checks = True
images_per_group = 8  # Per diagnosis and source group per epoch. None disables the cap.
run_feature_baseline = False
predict_official_images = True

batch_size = 32
number_of_epochs = 20
warmup_epochs = 2
warmup_learning_rate = 0.001
backbone_learning_rate = 0.00005
head_learning_rate = 0.0003
early_stopping_patience = 4
learning_rate_patience = 1
learning_rate_factor = 0.5
minimum_improvement = 0.0001
dropout_rate = 0.40
label_smoothing = 0.05
weight_decay = 0.001
seed = 42

selected_diseases = [
    "acne", "basal cell carcinoma", "bullous pemphigoid", "cellulitis",
    "eczema", "folliculitis", "herpes zoster", "impetigo", "lichen planus",
    "melanoma", "molluscum contagiosum", "nevus", "pityriasis rosea",
    "psoriasis", "rosacea", "scabies", "seborrheic keratoses",
    "squamous cell carcinoma", "urticaria", "vitiligo"
]

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Using:", device)

## 3. Get the Derm1M dataset

- **Full download.** Set `download_dataset_from_hugging_face = True`. This fetches
  the two CSVs and nine ZIP archives used by the notebook from the same revision,
  then extracts the referenced images under `hf_dataset_directory` in a folder
  named for that revision's commit. Hugging Face caches the downloaded archives.
  The first run needs about 35 GB for downloads plus space for the extracted
  images. If both download options are on, the full download takes precedence.
- **CSV download.** Set only `download_csvs_from_hugging_face = True` if you have
  already extracted the image folders beside the notebook.
- **Local files.** With both options off, keep `Derm1M_v2_pretrain.csv`,
  `Derm1M_v2_validation.csv`, and the image folders beside the notebook.

Derm1M is gated. Accept its access conditions in your Hugging Face account and
use the optional login cell with a read token if you have not signed in to the
runtime. Set `hf_dataset_revision` to a commit ID to pin a specific version.

The first full-download run builds an image audit for that revision. It can
reuse that audit on later runs. If neither saved comparison manifest is
present, it makes a fresh group-aware split.


In [ ]:
from pathlib import Path
from shutil import copyfileobj
from tempfile import TemporaryDirectory
from zipfile import ZipFile

training_csv_path = Path("Derm1M_v2_pretrain.csv")
official_csv_path = Path("Derm1M_v2_validation.csv")
image_root = Path(".")
csv_source = "local files"


def extract_image_archive(archive_path, destination, filenames):
    if destination.is_dir():
        missing = next((name for name in filenames if not (destination / name).is_file()), None)
        if missing is None:
            return
        raise ValueError(f"{destination} is incomplete (missing {missing}). Move it aside and rerun.")

    # Publish the directory only after every manifest image is extracted.
    with ZipFile(archive_path) as archive, TemporaryDirectory(dir=destination.parent) as temporary:
        members = {entry.filename.replace("\\", "/"): entry
                   for entry in archive.infolist() if not entry.is_dir()}
        staged = Path(temporary) / destination.name
        for name in tqdm(filenames, desc=f"Extracting {archive_path.name}", unit="image"):
            member = members.get(f"{destination.name}/{name}") or members.get(name)
            if member is None:
                raise ValueError(f"{archive_path.name} is missing the manifest image {name}.")
            target = staged / name
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as incoming, target.open("wb") as outgoing:
                copyfileobj(incoming, outgoing)
        staged.rename(destination)


if download_dataset_from_hugging_face:
    from huggingface_hub import snapshot_download

    image_folders = ("IIYI", "edu", "note", "public", "pubmed", "reddit",
                     "twitter", "validation_data", "youtube")
    archive_names = [f"{folder}.zip" for folder in image_folders]
    snapshot = Path(snapshot_download(
        repo_id=hf_dataset_repo_id, repo_type="dataset", revision=hf_dataset_revision,
        allow_patterns=[training_csv_path.name, official_csv_path.name, *archive_names],
        max_workers=2,
    ))
    training_csv_path = snapshot / training_csv_path
    official_csv_path = snapshot / official_csv_path
    image_root = Path(hf_dataset_directory).expanduser().resolve() / snapshot.name
    csv_source = f"Hugging Face ({snapshot.name})"
elif download_csvs_from_hugging_face:
    from huggingface_hub import hf_hub_download

    training_csv_path, official_csv_path = [
        hf_hub_download(repo_id=hf_dataset_repo_id, repo_type="dataset",
                        filename=path.name, revision=hf_dataset_revision)
        for path in (training_csv_path, official_csv_path)
    ]
    csv_source = "Hugging Face (CSVs only)"

raw_data = pd.read_csv(training_csv_path, low_memory=False)
official_data = pd.read_csv(official_csv_path, low_memory=False)

if download_dataset_from_hugging_face:
    images_by_folder = {folder: [] for folder in image_folders}
    manifest_images = pd.concat([raw_data["filename"], official_data["filename"]])
    manifest_images = manifest_images.str.strip().str.replace("\\", "/", regex=False)
    for filename in manifest_images.dropna().unique():
        folder, _, name = filename.partition("/")
        if folder not in images_by_folder or ":" in name or any(
                part in {"", ".", ".."} for part in name.split("/")):
            raise ValueError(f"Unexpected Derm1M manifest path: {filename!r}")
        images_by_folder[folder].append(name)
    image_root.mkdir(parents=True, exist_ok=True)
    for folder, filenames in images_by_folder.items():
        if filenames:
            extract_image_archive(snapshot / f"{folder}.zip", image_root / folder, filenames)

print("CSV source:", csv_source)
print("Image directory:", image_root)
print("Training rows:", len(raw_data))
print("Official rows:", len(official_data))
print(raw_data.columns.tolist())
raw_data.head()


## 4. Clean the labels

Keep one clear diagnosis per image. Do not merge similar disease names automatically.
Remove conflicting labels for the same filename and captions that explicitly describe
an uncertain diagnosis. This text screen can reject valid examples; it does not prove
that the remaining labels are medically correct. Captions are never model inputs.


In [ ]:
def clean_label(value):
    labels = [word.strip() for word in value.split(",")]
    if len(set(labels)) != 1:
        return None
    if labels[0] in ["", "nan", "none", "unknown", "no definitive diagnosis"]:
        return None
    return labels[0]


data = raw_data[["filename", "disease_label", "source", "caption"]].fillna("").copy()
data["filename"] = data["filename"].str.strip().str.replace("\\", "/", regex=False)
data["disease_label"] = data["disease_label"].str.lower().str.split().str.join(" ")
data["label"] = data["disease_label"].apply(clean_label)

label_counts = data.groupby("filename")["disease_label"].nunique()
conflicting_files = label_counts[label_counts > 1].index
data = data[~data["filename"].isin(conflicting_files)]
data = data.dropna(subset=["label"])
data = data[data["filename"] != ""]
data = data.drop_duplicates(subset="filename")

uncertain_words = r"differential diagnos(?:is|es)|differential includes|suspected|possible diagnosis|uncertain diagnosis|rule out"
uncertain = data["caption"].str.contains(r"\b(?:" + uncertain_words + r")\b", case=False, regex=True)
data = data[~uncertain]
data = data[["filename", "label", "source"]].reset_index(drop=True)
print("Single-label images after cleaning:", len(data))

## 5. Keep related images together

Images from the same video, article, post, or textbook page should stay in the same
split. These groups are useful clues, but they cannot guarantee different patients.
Also exclude training groups that occur in the official data.


In [ ]:
official_groups = official_data[["filename", "source"]].copy()

for table in [data, official_groups]:
    names = table["filename"].str.split("/").str[-1]
    stems = names.str.rsplit(".", n=1).str[0]
    table["group"] = "image:" + names

    youtube_rows = table["source"].eq("youtube")
    table.loc[youtube_rows, "group"] = "youtube:" + stems[youtube_rows].str.split("_frame_").str[0]

    article_ids = stems.str.extract(r"(PMC\d+)", flags=re.IGNORECASE, expand=False)
    table.loc[article_ids.notna(), "group"] = "pubmed:" + article_ids.dropna().str.lower()

    for website in ["iiyi", "reddit", "twitter"]:
        website_rows = table["source"].str.lower().str.startswith(website)
        table.loc[website_rows, "group"] = website + ":" + stems[website_rows].str.split("_").str[0]

    pages = stems.str.extract(r"^(.+?_\d{5})(?:_.*)?$", expand=False)
    textbook_rows = table["source"].str.startswith("textbook") & pages.notna()
    table.loc[textbook_rows, "group"] = "book:" + pages[textbook_rows]

data = data[~data["group"].isin(official_groups["group"])].reset_index(drop=True)
print("Images after separating official groups:", len(data))

## 6. Read images in a consistent color space

We need one image-reading function because training, validation, and prediction must
handle colors in the same way. It corrects rotation, converts valid color profiles to
sRGB, and puts transparent pixels on white. Untagged RGB images are assumed to be
sRGB. Unsupported color cases raise an error for review.


In [ ]:
def load_image(filename):
    with Image.open(image_root / filename) as original:
        if getattr(original, "n_frames", 1) != 1:
            raise ValueError("This image has multiple frames.")
        image = ImageOps.exif_transpose(original)
        if image.mode not in ["1", "L", "LA", "P", "RGB", "RGBA", "CMYK", "LAB", "YCbCr"]:
            raise ValueError("This image mode needs review: " + image.mode)

        alpha = None
        if "A" in image.getbands() or "transparency" in image.info:
            rgba_image = image.convert("RGBA")
            alpha = rgba_image.getchannel("A")
        color_image = rgba_image if alpha is not None else image
        profile_bytes = image.info.get("icc_profile")

        if profile_bytes:
            profile = ImageCms.ImageCmsProfile(io.BytesIO(profile_bytes))
            profile_space = profile.profile.xcolor_space.strip()
            if profile_space == "RGB" and image.mode not in ["CMYK", "LAB"]:
                color_image = color_image.convert("RGB")
            elif profile_space == "GRAY" and image.mode in ["1", "L", "LA"]:
                color_image = image.convert("L")
            elif not ((profile_space == "CMYK" and image.mode == "CMYK") or
                      (profile_space == "Lab" and image.mode == "LAB")):
                raise ValueError("The image mode and color profile do not match.")
            color_image = ImageCms.profileToProfile(
                color_image, profile, ImageCms.createProfile("sRGB"), outputMode="RGB")
        else:
            exif = image.getexif()
            color_tag = exif.get_ifd(34665).get(40961, exif.get(40961))
            if image.mode in ["CMYK", "LAB"] or color_tag == 2:
                raise ValueError("This image needs a color profile.")
            color_image = color_image.convert("RGB")

        if alpha is not None:
            color_image = Image.composite(color_image, Image.new("RGB", image.size, "white"), alpha)
        return color_image

## 7. Check image quality and remove exact copies

The existing audit CSV contains image sizes, modes, color decisions, and pixel
fingerprints. Reuse it for unchanged images. The alternative loop below does the
checks inside this notebook, which takes longer.

Check duplicates across all remaining labels before selecting the 20 diagnoses.
Remove tiny/unreadable images, conflicting copies, and copies of official images.
Keep one representative when duplicate pixels have the same label. Fingerprints
catch exact displayed copies; they do not catch every cropped or compressed copy.


In [ ]:
audit_path = image_root / "image_color_audit.csv"
if reuse_image_checks and (not download_dataset_from_hugging_face or audit_path.is_file()):
    image_checks = pd.read_csv(audit_path, keep_default_na=False)
else:
    filenames = pd.concat([data["filename"], official_data["filename"]]).drop_duplicates()
    checked_images = []
    for filename in tqdm(filenames, desc="Checking images"):
        result = {"filename": filename, "status": "ok", "error": "", "width": 0,
                  "height": 0, "original_mode": "", "pixel_hash": ""}
        try:
            with Image.open(image_root / filename) as original:
                result["original_mode"] = original.mode
            image = load_image(filename)
            result["width"], result["height"] = image.size
            result["pixel_hash"] = hashlib.sha256(str(image.size).encode() + image.tobytes()).hexdigest()
        except (OSError, ValueError, TypeError, SyntaxError, ImageCms.PyCMSError, Image.DecompressionBombError) as error:
            result["status"] = "needs_review"
            result["error"] = str(error)
        checked_images.append(result)
    image_checks = pd.DataFrame(checked_images)
    if download_dataset_from_hugging_face:
        with TemporaryDirectory(dir=image_root) as temporary:
            staged_audit = Path(temporary) / audit_path.name
            image_checks.to_csv(staged_audit, index=False)
            staged_audit.replace(audit_path)
    else:
        image_checks.to_csv(run_name + "_image_checks.csv", index=False)

print("Original image modes:")
print(image_checks["original_mode"].value_counts())
image_checks = image_checks[["filename", "status", "width", "height", "pixel_hash"]]
data = data.merge(image_checks, on="filename", how="left", validate="one_to_one")
official_checks = official_data[["filename"]].merge(image_checks, on="filename", how="left", validate="one_to_one")
if data["status"].isna().any() or official_checks["status"].isna().any():
    raise ValueError("The audit is incomplete. Set reuse_image_checks=False and rerun from the CSV cell.")

data = data[(data["status"] == "ok") & (data["width"] >= 64) & (data["height"] >= 64)]
labels_per_image = data.groupby("pixel_hash")["label"].nunique()
conflicting_hashes = labels_per_image[labels_per_image > 1].index
official_hashes = official_checks.loc[official_checks["status"] == "ok", "pixel_hash"]
data = data[~data["pixel_hash"].isin(conflicting_hashes)]
data = data[~data["pixel_hash"].isin(official_hashes)]
data = data.sort_values("filename").drop_duplicates("pixel_hash").sort_index().reset_index(drop=True)
print("Cleaned single-label images:", len(data))
print("Official images needing review:", (official_checks["status"] != "ok").sum())

## 8. Select the 20 diagnoses

Check how many images, groups, and sources each diagnosis has before training.


In [ ]:
data = data[data["label"].isin(selected_diseases)].reset_index(drop=True)
class_names = sorted(selected_diseases)
label_numbers = {name: number for number, name in enumerate(class_names)}

class_counts = pd.DataFrame({
    "images": data.groupby("label").size(),
    "groups": data.groupby("label")["group"].nunique(),
    "sources": data.groupby("label")["source"].nunique()
}).reindex(class_names).fillna(0)
if (class_counts["images"] < 150).any() or (class_counts["groups"] < 50).any() or (class_counts["sources"] < 2).any():
    raise ValueError("Some diagnoses have too little usable data. Check class_counts before training.")
class_counts

## 9. Reuse the comparison split or create a new one

Each saved filename, diagnosis, source group, and pixel hash must still match
the cleaned data. Training and validation stay separate by group and exact
pixels. If the reference is empty, make a fresh group-aware split. The full
Hugging Face download also creates a split when neither reference file exists.

Within training, use a seeded ordering within each diagnosis and source group.
The epoch cap takes consecutive images in that ordering and wraps around, so
large groups contribute different images across epochs. Validation keeps its
original distribution. Fast mode uses a fixed validation subset during
training, then evaluates the full validation set.


In [ ]:
from pathlib import Path

use_reference_split = bool(split_reference_run)
split_paths = [Path(split_reference_run + suffix) for suffix in ("_training.csv", "_validation.csv")]
if download_dataset_from_hugging_face and use_reference_split and not any(path.exists() for path in split_paths):
    print("No local comparison manifests found. Creating a new group-aware split.")
    use_reference_split = False

if use_reference_split:
    split_tables = []
    split_columns = ["filename", "label", "group", "pixel_hash"]
    current_by_filename = data.set_index("filename", verify_integrity=True)
    for manifest_path in split_paths:
        if not manifest_path.is_file():
            raise ValueError("Missing reference split: " + str(manifest_path) +
                             ". Copy it beside the notebook or explicitly clear split_reference_run.")
        saved_split = pd.read_csv(manifest_path)
        if saved_split["filename"].duplicated().any():
            raise ValueError("Reference split contains duplicate filenames: " + str(manifest_path))
        if not set(saved_split["filename"]).issubset(current_by_filename.index):
            raise ValueError("Reference split contains images absent from the cleaned data.")
        current_split = current_by_filename.loc[saved_split["filename"]].reset_index()
        saved_values = saved_split[split_columns].astype("string").reset_index(drop=True)
        current_values = current_split[split_columns].astype("string").reset_index(drop=True)
        if not saved_values.equals(current_values):
            raise ValueError("Reference labels, groups, or image hashes changed. Review the data first.")
        split_tables.append(current_split)
    training_pool, full_validation_data = split_tables
    if set(training_pool["filename"]) & set(full_validation_data["filename"]):
        raise ValueError("Reference training and validation images overlap.")
    if set(training_pool["filename"]) | set(full_validation_data["filename"]) != set(data["filename"]):
        raise ValueError("The cleaned dataset changed. Review it before reusing the reference split.")
    print("Using saved split from:", split_reference_run)
else:
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    train_indices, validation_indices = next(splitter.split(data, data["label"], data["group"]))
    training_pool = data.iloc[train_indices].reset_index(drop=True)
    full_validation_data = data.iloc[validation_indices].reset_index(drop=True)

assert not set(training_pool["group"]) & set(full_validation_data["group"])
assert not set(training_pool["pixel_hash"]) & set(full_validation_data["pixel_hash"])
assert set(training_pool["label"]) == set(class_names)
assert set(full_validation_data["label"]) == set(class_names)

if images_per_group is not None:
    if not isinstance(images_per_group, int) or images_per_group < 1:
        raise ValueError("images_per_group must be a positive integer or None.")
    group_order = training_pool.sample(frac=1, random_state=seed).reset_index(drop=True)
    training_groups = group_order.groupby(["label", "group"])["filename"]
    group_positions = training_groups.cumcount()
    group_sizes = training_groups.transform("size")


def training_rows_for_epoch(epoch):
    rows = training_pool
    if images_per_group is not None:
        offset = epoch * images_per_group
        rows = group_order[(group_positions - offset) % group_sizes < images_per_group]
    if not fast_mode:
        return rows

    samples = []
    for _, disease_rows in rows.groupby("label", sort=True):
        disease_rows = disease_rows.sample(frac=1, random_state=seed)
        # Prioritize one image per group before taking additional images.
        image_numbers = disease_rows.groupby("group").cumcount()
        order = image_numbers.sort_values(kind="stable").index
        disease_rows = disease_rows.loc[order]
        sample_size = min(images_per_class, len(disease_rows))
        start = (epoch * sample_size) % len(disease_rows)
        samples.append(pd.concat([disease_rows.iloc[start:], disease_rows.iloc[:start]]).head(sample_size))
    return pd.concat(samples, ignore_index=True)


training_data = training_rows_for_epoch(0)
validation_data = full_validation_data
if fast_mode:
    validation_parts = [
        rows.sample(n=min(validation_images_per_class, len(rows)), random_state=seed)
        for _, rows in full_validation_data.groupby("label", sort=True)
    ]
    validation_data = pd.concat(validation_parts, ignore_index=True)

print("Training images available per epoch:", len(training_data), "out of", len(training_pool))
print("Validation images per epoch:", len(validation_data), "out of", len(full_validation_data))


## 10. Denoise, resize, and normalize

All model inputs use the same RGB conversion, aspect-ratio-preserving resize,
and normalization. An optional mild bilateral filter smooths small pixel
variations at model resolution while limiting smoothing across stronger
edges. Inspect the examples below for changes to color and fine texture.
The original files and their duplicate fingerprints stay intact.

Mean-color padding keeps the entire frame visible. Normalization uses the
channel means and standard deviations of the selected ImageNet weights.
Flips, small rotations, and mild brightness and contrast changes apply only
during training.


In [ ]:
pretrained_weights = {
    "resnet18": ResNet18_Weights.DEFAULT,
    "swin_t": Swin_T_Weights.DEFAULT,
}.get(model_name)
if pretrained_weights is None:
    raise ValueError('model_name must be "resnet18" or "swin_t".')
pretrained_transform = pretrained_weights.transforms()
image_size = pretrained_transform.crop_size[0]
padding_color = tuple(round(channel * 255) for channel in pretrained_transform.mean)


def resize_image(image, apply_denoise):
    longest_side = max(image.size)
    size = tuple(max(1, round(side / longest_side * image_size)) for side in image.size)
    resized = image.resize(size, resample=Image.Resampling.BILINEAR)
    if apply_denoise:
        pixels = np.asarray(resized)
        resized = Image.fromarray(cv2.bilateralFilter(pixels, d=5, sigmaColor=18, sigmaSpace=1.5))
    canvas = Image.new("RGB", (image_size, image_size), padding_color)
    offset = tuple(round((image_size - side) / 2) for side in size)
    canvas.paste(resized, offset)
    return canvas


def preprocess_image(image):
    return resize_image(image, apply_denoise=denoise_images)


def check_checkpoint_preprocessing(checkpoint):
    if checkpoint.get("denoise_images", False) != denoise_images:
        raise ValueError("Match denoise_images to the saved checkpoint before using it.")


normalization = transforms.Normalize(pretrained_transform.mean, pretrained_transform.std)
validation_transform = transforms.Compose([preprocess_image, transforms.ToTensor(), normalization])
training_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10, interpolation=transforms.InterpolationMode.BILINEAR,
                              expand=True, fill=padding_color),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    validation_transform,
])

examples = training_data.groupby("label").sample(n=1, random_state=seed)
examples = examples.sample(n=min(4, len(examples)), random_state=seed)
figure, axes = plt.subplots(2, len(examples), squeeze=False, figsize=(10, 5))
for column, row in enumerate(examples.itertuples()):
    image = load_image(row.filename)
    axes[0, column].imshow(resize_image(image, apply_denoise=False))
    axes[0, column].set_title(row.label, fontsize=9)
    axes[1, column].imshow(resize_image(image, apply_denoise=True))
figure.suptitle("Resized (top) and resized with optional denoising (bottom)")
for axis in axes.flat:
    axis.axis("off")
plt.tight_layout()
plt.show()


## 11. Put images into batches

Both architectures use the same small Dataset class to load images one at a time. It returns the
image and its class number. A DataLoader combines these into batches. Zero worker
processes keeps this easy to run in local notebooks.


In [ ]:
class SkinDataset(Dataset):
    def __init__(self, rows, transform):
        self.rows = rows.reset_index(drop=True)
        self.transform = transform
        self.label_numbers = label_numbers.copy()

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        image = self.transform(load_image(row["filename"]))
        label = self.label_numbers[row["label"]]
        return image, label


validation_loader = DataLoader(SkinDataset(validation_data, validation_transform), batch_size=batch_size)
full_validation_loader = DataLoader(SkinDataset(full_validation_data, validation_transform), batch_size=batch_size)
print("Training batches per epoch:", (len(training_data) + batch_size - 1) // batch_size)
print("Validation batches per epoch:", len(validation_loader))

## 12. Optional: try simple feature engineering

This baseline uses 12 global color and texture measurements. Fit the scaler on
training images only. These features include the background and are not a diagnosis
rule. The CNN learns its own features separately. Enable `run_feature_baseline` to
run this comparison; it uses the first training sample.


In [ ]:
def image_features(image):
    pixels = np.asarray(image.resize((64, 64)), dtype=np.float32) / 255.0
    gray = pixels[:, :, 0] * 0.299 + pixels[:, :, 1] * 0.587 + pixels[:, :, 2] * 0.114
    saturation = (pixels.max(axis=2) - pixels.min(axis=2)) / np.maximum(pixels.max(axis=2), 0.000001)
    return np.concatenate([
        pixels.mean(axis=(0, 1)), pixels.std(axis=(0, 1)),
        [gray.mean(), gray.std(), saturation.mean(), saturation.std(),
         np.abs(np.diff(gray, axis=0)).mean(), np.abs(np.diff(gray, axis=1)).mean()]
    ])


if run_feature_baseline:
    training_features = [image_features(load_image(name)) for name in tqdm(training_data["filename"])]
    validation_features = [image_features(load_image(name)) for name in tqdm(validation_data["filename"])]
    scaler = StandardScaler()
    training_features = scaler.fit_transform(training_features)
    validation_features = scaler.transform(validation_features)
    baseline = LogisticRegression(max_iter=1000, class_weight="balanced")
    baseline.fit(training_features, training_data["label"])
    baseline_predictions = baseline.predict(validation_features)
    print("Feature baseline accuracy:", accuracy_score(validation_data["label"], baseline_predictions))
    print("Feature baseline macro F1:", f1_score(validation_data["label"], baseline_predictions, labels=class_names, average="macro", zero_division=0))

## 13. Write the validation step

This function is reused after each epoch and for the final report. It turns off
training behavior and gradients, then returns the true classes, predicted classes,
and average loss. Validation uses ordinary, unweighted cross-entropy.


In [ ]:
def evaluate_model(model, loader):
    model.eval()
    actual_labels = []
    predicted_labels = []
    total_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validation", unit="batch", leave=False):
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            total_loss += nn.functional.cross_entropy(outputs, labels, reduction="sum").item()
            actual_labels.extend(labels.cpu().tolist())
            predicted_labels.extend(outputs.argmax(dim=1).cpu().tolist())
    return actual_labels, predicted_labels, total_loss / len(actual_labels)

## 14. Build or resume the chosen model and train it

The classifier learns alone for two epochs. After warmup, ResNet18 also trains
`layer4`; Swin-Tiny also trains `features[-1]` and the final LayerNorm. Earlier
stages stay frozen, and ResNet BatchNorm running statistics stay fixed.
The classifier uses dropout and the training loss uses label smoothing.
Validation uses unweighted cross-entropy with the original labels.

The default shuffles the capped epoch pool without replacement and uses
square-root class weights calculated from that pool. Optional balanced
sampling draws with replacement using inverse class frequency and uses
unweighted loss. Training accuracy describes the augmented training draws.

The scheduler reduces the learning rates when validation macro F1 plateaus.
Early stopping uses the same metric. The report also tracks image accuracy
and group-weighted accuracy, in which each source group counts equally.

For a new experiment, use an unused run name. To resume, keep the original
settings and split, set `resume_training = True`, and raise the total epoch
limit if needed. The notebook backs up the saved model and history and
resumes from the best checkpoint, which can precede the last logged epoch.
Resume checks the training and preprocessing settings before loading weights.
For checkpoints from before these changes, match the original settings,
including `denoise_images = False`, `images_per_group = None`,
`label_smoothing = 0.0`, and `learning_rate_factor = 0.1`.
Random augmentation is reseeded on restart.


In [ ]:
from pathlib import Path
from shutil import copy2
from datetime import datetime, timezone

model_filename = run_name + "_model.pth"
history_filename = run_name + "_history.csv"
resume_checkpoint = None
start_epoch = 0
history = []

if min(warmup_learning_rate, backbone_learning_rate, head_learning_rate) <= 0:
    raise ValueError("Learning rates must be positive.")

if not 0 <= warmup_epochs < number_of_epochs:
    raise ValueError("warmup_epochs must be smaller than number_of_epochs.")
if learning_rate_patience < 0 or early_stopping_patience < learning_rate_patience + 2:
    raise ValueError("Early stopping must allow an epoch after the learning rate decreases.")

if not 0 <= label_smoothing < 1 or not 0 < learning_rate_factor < 1:
    raise ValueError("Check the label smoothing and learning-rate reduction settings.")

training_settings = {
    "dropout": dropout_rate,
    "weight_decay": weight_decay,
    "warmup_epochs": warmup_epochs,
    "fast_mode": fast_mode,
    "images_per_class": images_per_class,
    "validation_images_per_class": validation_images_per_class,
    "batch_size": batch_size,
    "seed": seed,
    "learning_rate_patience": learning_rate_patience,
    "early_stopping_patience": early_stopping_patience,
    "minimum_improvement": minimum_improvement,
    "warmup_learning_rate": warmup_learning_rate,
    "backbone_learning_rate": backbone_learning_rate,
    "head_learning_rate": head_learning_rate,
    "learning_rate_factor": learning_rate_factor,
    "denoise_images": denoise_images,
    "images_per_group": images_per_group,
    "label_smoothing": label_smoothing,
}
legacy_settings = {
    "validation_images_per_class": 50,
    "learning_rate_factor": 0.1,
    "denoise_images": False,
    "images_per_group": None,
    "label_smoothing": 0.0,
}

if resume_training:
    if not Path(model_filename).is_file():
        raise ValueError("No saved model for this run. Set resume_training=False to start it.")
    resume_checkpoint = torch.load(model_filename, map_location="cpu", weights_only=True)
    if resume_checkpoint.get("model_name", "resnet18") != model_name:
        raise ValueError("This checkpoint uses a different model. Restore model_name or use a new run_name.")
    if resume_checkpoint.get("balanced_sampling", False) != balanced_sampling:
        raise ValueError("This checkpoint uses a different sampling mode. Use a new run_name to change it.")
    if resume_checkpoint["classes"] != class_names:
        raise ValueError("Saved classes differ. Use the original classes or choose a new run_name.")
    for setting, value in training_settings.items():
        if resume_checkpoint.get(setting, legacy_settings.get(setting, value)) != value:
            raise ValueError("Resume requires the saved " + setting + ". Use a new run_name to change it.")
    start_epoch = int(resume_checkpoint["epoch"])
    if start_epoch >= number_of_epochs:
        raise ValueError("Increase number_of_epochs above the saved checkpoint epoch to continue.")
    if not Path(history_filename).is_file():
        raise ValueError("Resume needs this run's history CSV to preserve its training history.")
    saved_history = pd.read_csv(history_filename)
    saved_history = saved_history[saved_history["epoch"] <= start_epoch]
    if saved_history["epoch"].tolist() != list(range(1, start_epoch + 1)):
        raise ValueError("The history CSV does not match the saved checkpoint epochs.")
    if not np.isclose(saved_history.iloc[-1]["macro_f1"], resume_checkpoint["macro_f1"]):
        raise ValueError("The history CSV does not match the saved checkpoint score.")
    history = saved_history.to_dict("records")
    for suffix, current_rows in [("_training.csv", training_pool),
                                  ("_validation.csv", full_validation_data)]:
        manifest = Path(run_name + suffix)
        if not manifest.is_file():
            raise ValueError("Resume needs the original split file: " + str(manifest))
        saved_rows = pd.read_csv(manifest)
        columns = ["filename", "label", "group", "pixel_hash"]
        # These fields are text; CSV loading may infer a different Pandas text dtype.
        saved_rows = saved_rows[columns].astype("string").sort_values("filename").reset_index(drop=True)
        current_rows = current_rows[columns].astype("string").sort_values("filename").reset_index(drop=True)
        if not saved_rows.equals(current_rows):
            raise ValueError("The data split changed. Restore it or use a new run_name.")
    print("Continuing from best checkpoint at epoch", start_epoch,
          "with macro F1", round(resume_checkpoint["macro_f1"], 4))
elif Path(model_filename).exists() or Path(history_filename).exists():
    raise ValueError("This run already exists. Set resume_training=True or choose a new run_name.")

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if model_name == "resnet18":
    model = resnet18(weights=None if resume_training else pretrained_weights)
    model.fc = nn.Sequential(nn.Dropout(dropout_rate), nn.Linear(model.fc.in_features, len(class_names)))
    classifier = model.fc
    finetuning_modules = [model.layer4]
else:
    model = swin_t(weights=None if resume_training else pretrained_weights)
    model.head = nn.Sequential(nn.Dropout(dropout_rate), nn.Linear(model.head.in_features, len(class_names)))
    classifier = model.head
    finetuning_modules = [model.features[-1], model.norm]
if resume_checkpoint is not None:
    model.load_state_dict(resume_checkpoint["weights"])
model = model.to(device)
for parameter in model.parameters():
    parameter.requires_grad = False
for parameter in classifier.parameters():
    parameter.requires_grad = True

counts = training_rows_for_epoch(0)["label"].value_counts().reindex(class_names).to_numpy()
if balanced_sampling:
    class_weights = torch.ones(len(class_names), dtype=torch.float32, device=device)
    loss_function = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
else:
    class_weights = np.sqrt(counts.sum() / (len(class_names) * counts))
    class_weights = torch.tensor(class_weights / class_weights.mean(), dtype=torch.float32, device=device)
    loss_function = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)
optimizer = torch.optim.AdamW(classifier.parameters(), lr=warmup_learning_rate, weight_decay=weight_decay)
print("Model:", model_name, "| Balanced sampling:", balanced_sampling,
      "| Training loss:", "unweighted" if balanced_sampling else "class-weighted")

best_f1 = resume_checkpoint["macro_f1"] if resume_checkpoint is not None else -1.0
stopping_best = resume_checkpoint.get("stopping_best", best_f1) if resume_checkpoint is not None else -1.0
epochs_without_improvement = resume_checkpoint.get("epochs_without_improvement", 0) if resume_checkpoint is not None else 0


def make_finetuning_optimizer():
    backbone_parameters = []
    for module in finetuning_modules:
        for parameter in module.parameters():
            parameter.requires_grad = True
            backbone_parameters.append(parameter)
    return torch.optim.AdamW([
        {"params": backbone_parameters, "lr": backbone_learning_rate},
        {"params": classifier.parameters(), "lr": head_learning_rate}
    ], weight_decay=weight_decay)


def make_scheduler(optimizer):
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=learning_rate_factor, patience=learning_rate_patience,
        threshold=minimum_improvement, threshold_mode="abs")


def sampled_training_rows(epoch):
    available_rows = training_rows_for_epoch(epoch)
    if not balanced_sampling:
        return available_rows
    class_counts = available_rows["label"].value_counts()
    sample_weights = available_rows["label"].map(1.0 / class_counts).to_numpy(dtype=np.float64, copy=True)
    generator = torch.Generator().manual_seed(seed + epoch)
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(available_rows),
                                    replacement=True, generator=generator)
    return available_rows.iloc[list(sampler)].reset_index(drop=True)


def set_training_mode():
    # Keep frozen Swin stochastic-depth layers inactive as well as frozen BatchNorm.
    model.eval()
    classifier.train()
    if scheduler is not None:
        for module in finetuning_modules:
            module.train()
    for layer in model.modules():
        if isinstance(layer, nn.BatchNorm2d):
            layer.eval()


scheduler = None
if resume_checkpoint is not None:
    if start_epoch > warmup_epochs:
        optimizer = make_finetuning_optimizer()
        scheduler = make_scheduler(optimizer)
    if "optimizer_state" in resume_checkpoint:
        optimizer.load_state_dict(resume_checkpoint["optimizer_state"])
        if scheduler is not None:
            scheduler.load_state_dict(resume_checkpoint["scheduler_state"])
    else:
        head_lr = float(history[-1]["head_learning_rate"])
        if scheduler is None:
            optimizer.param_groups[0]["lr"] = head_lr
        else:
            optimizer.param_groups[0]["lr"] = head_lr * (backbone_learning_rate / head_learning_rate)
            optimizer.param_groups[1]["lr"] = head_lr
            scheduler.step(stopping_best)
        print("Legacy checkpoint: weights and recorded learning rates restored;",
              "optimizer, scheduler, and early-stopping state restart.")

seen_images = set()
if resume_training:
    if "seen_images" in resume_checkpoint:
        seen_images.update(resume_checkpoint["seen_images"])
    elif fast_mode or balanced_sampling:
        for previous_epoch in range(start_epoch):
            seen_images.update(sampled_training_rows(previous_epoch)["filename"])
    else:
        seen_images.update(training_pool["filename"])
    backup_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    for filename in [model_filename, history_filename]:
        file_path = Path(filename)
        backup_path = file_path.with_name(file_path.stem + "_before_resume_" + backup_stamp + file_path.suffix)
        copy2(file_path, backup_path)
        print("Backup saved:", backup_path)
else:
    training_pool.to_csv(run_name + "_training.csv", index=False)
    full_validation_data.to_csv(run_name + "_validation.csv", index=False)

validation_group_weights = 1.0 / validation_data.groupby("group")["filename"].transform("size").to_numpy()

for epoch in range(start_epoch, number_of_epochs):
    start_time = time.perf_counter()
    training_data = sampled_training_rows(epoch)
    training_loader = DataLoader(SkinDataset(training_data, training_transform), batch_size=batch_size,
                                 shuffle=not balanced_sampling,
                                 generator=torch.Generator().manual_seed(seed + epoch))
    if balanced_sampling:
        drawn_counts = training_data["label"].value_counts().reindex(class_names, fill_value=0)
        print("Balanced draws per class: min", int(drawn_counts.min()), "max", int(drawn_counts.max()))

    if epoch == warmup_epochs and scheduler is None:
        optimizer = make_finetuning_optimizer()
        scheduler = make_scheduler(optimizer)
        if stopping_best >= 0:
            scheduler.step(stopping_best)

    set_training_mode()
    correct_predictions = 0
    total_images = 0
    total_loss = 0.0
    total_weight = 0.0
    for images, labels in tqdm(training_loader, desc=f"Epoch {epoch + 1}: training", unit="batch"):
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, labels)
        if not torch.isfinite(loss):
            raise ValueError("Training loss is invalid. Check the images and learning rate.")
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        batch_weight = class_weights[labels].sum().item()
        total_loss += loss.item() * batch_weight
        total_weight += batch_weight
        correct_predictions += (outputs.argmax(dim=1) == labels).sum().item()
        total_images += len(labels)

    actual, predicted, validation_loss = evaluate_model(model, validation_loader)
    validation_accuracy = accuracy_score(actual, predicted)
    group_accuracy = accuracy_score(actual, predicted, sample_weight=validation_group_weights)
    macro_f1 = f1_score(actual, predicted, labels=range(len(class_names)), average="macro", zero_division=0)
    seen_images.update(training_data["filename"])
    history.append({"epoch": epoch + 1, "training_loss": total_loss / total_weight,
                    "training_accuracy": correct_predictions / total_images,
                    "validation_loss": validation_loss, "validation_accuracy": validation_accuracy,
                    "macro_f1": macro_f1, "validation_group_accuracy": group_accuracy,
                    "distinct_training_images": len(seen_images),
                    "head_learning_rate": optimizer.param_groups[-1]["lr"],
                    "backbone_learning_rate": optimizer.param_groups[0]["lr"] if scheduler is not None else 0.0,
                    "seconds": time.perf_counter() - start_time})
    pd.DataFrame(history).to_csv(run_name + "_history.csv", index=False)
    print(f"Train accuracy: {correct_predictions / total_images:.3f} | "
          f"Validation accuracy: {validation_accuracy:.3f} | Group-weighted: {group_accuracy:.3f}")
    print(f"Validation loss: {validation_loss:.3f} | Macro F1: {macro_f1:.3f} | Classes predicted: {len(set(predicted))}/{len(class_names)}")

    improved_checkpoint = macro_f1 > best_f1
    if epoch >= warmup_epochs:
        scheduler.step(macro_f1)
    if macro_f1 > stopping_best + minimum_improvement:
        stopping_best = macro_f1
        epochs_without_improvement = 0
    elif epoch >= warmup_epochs:
        epochs_without_improvement += 1
    if improved_checkpoint:
        best_f1 = macro_f1
        torch.save({
            **training_settings,
            "weights": model.state_dict(),
            "classes": class_names,
            "model_name": model_name,
            "balanced_sampling": balanced_sampling,
            "seen_images": sorted(seen_images),
            "epoch": epoch + 1,
            "macro_f1": macro_f1,
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
            "stopping_best": stopping_best,
            "epochs_without_improvement": epochs_without_improvement,
        }, model_filename)
    if epochs_without_improvement >= early_stopping_patience:
        print("Early stopping: validation macro F1 stopped improving.")
        break

## 15. Evaluate the best model

Reload the checkpoint with the highest validation macro F1 and report full
validation accuracy, macro F1, and group-weighted accuracy. Every diagnosis
gets equal weight in macro F1. Group-weighted accuracy gives each source
group equal weight even when it contains hundreds of images. This internal
validation set includes the data used for checkpoint selection.

The confusion matrix and exported errors help locate weak diagnoses and
sources. Review validation loss alongside accuracy to spot overfitting.


In [ ]:
checkpoint = torch.load(model_filename, map_location=device, weights_only=True)
assert checkpoint["classes"] == class_names
assert checkpoint.get("model_name", "resnet18") == model_name
check_checkpoint_preprocessing(checkpoint)
print("Model:", model_name, "| Balanced sampling:", balanced_sampling)
model.load_state_dict(checkpoint["weights"])
actual, predicted, validation_loss = evaluate_model(model, full_validation_loader)
print("Best epoch:", checkpoint["epoch"])
print("Full validation accuracy:", accuracy_score(actual, predicted))
print("Full validation macro F1:", f1_score(actual, predicted, labels=range(len(class_names)), average="macro", zero_division=0))
print("Full validation loss:", validation_loss)
full_group_weights = 1.0 / full_validation_data.groupby("group")["filename"].transform("size").to_numpy()
group_accuracy = accuracy_score(actual, predicted, sample_weight=full_group_weights)
print("Group-weighted validation accuracy:", group_accuracy)

report = classification_report(actual, predicted, labels=range(len(class_names)), target_names=class_names, output_dict=True, zero_division=0)
report_table = pd.DataFrame(report).T.drop(index="accuracy", errors="ignore")
report_table.to_csv(run_name + "_report.csv")
summary = pd.DataFrame([{
    "run_name": run_name, "model_name": model_name, "balanced_sampling": balanced_sampling,
    "denoise_images": denoise_images, "images_per_group": checkpoint.get("images_per_group"),
    "label_smoothing": checkpoint.get("label_smoothing", 0.0),
    "dropout": checkpoint["dropout"], "best_epoch": checkpoint["epoch"],
    "validation_images": len(actual), "validation_accuracy": accuracy_score(actual, predicted),
    "validation_group_accuracy": group_accuracy, "macro_f1": report_table.loc["macro avg", "f1-score"],
    "macro_recall": report_table.loc["macro avg", "recall"], "validation_loss": validation_loss
}])
summary.to_csv(run_name + "_summary.csv", index=False)
print(summary.to_string(index=False))
print(report_table)

predictions = full_validation_data.copy()
predictions["predicted_label"] = [class_names[number] for number in predicted]
predictions["correct"] = predictions["label"] == predictions["predicted_label"]
predictions.to_csv(run_name + "_validation_predictions.csv", index=False)
predictions[~predictions["correct"]].to_csv(run_name + "_mistakes.csv", index=False)
print(predictions.groupby("source")["correct"].agg(["count", "mean"]))
predictions.groupby(["source", "label"])["correct"].agg(["count", "mean"]).to_csv(run_name + "_source_scores.csv")

matrix = confusion_matrix(actual, predicted, labels=range(len(class_names)))
plt.figure(figsize=(14, 11))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, annot_kws={"fontsize": 7})
plt.xlabel("Predicted diagnosis")
plt.ylabel("Dataset diagnosis")
plt.tight_layout()
plt.show()

history_table = pd.DataFrame(history)
figure, axes = plt.subplots(1, 3, figsize=(14, 3))
axes[0].plot(history_table["epoch"], history_table["training_accuracy"], label="Augmented training")
axes[0].plot(history_table["epoch"], history_table["validation_accuracy"], label="Validation")
axes[0].set_title("Accuracy")
axes[0].legend()
axes[1].plot(history_table["epoch"], history_table["macro_f1"])
axes[1].set_title("Validation macro F1")
axes[2].plot(history_table["epoch"], history_table["validation_loss"])
axes[2].set_title("Validation loss")
plt.tight_layout()
plt.show()

## 16. Optional: predict the official images

Run the best-checkpoint cell before making predictions. The official CSV has no diagnosis labels. Keep one output row per input image,
including an image status if it could not be checked. Probabilities only compare
the 20 known classes; they are not calibrated medical confidence.


In [ ]:
if predict_official_images:
    check_checkpoint_preprocessing(checkpoint)
    official_results = official_data.copy()
    official_results["image_status"] = official_checks["status"].to_numpy()
    official_results["predicted_label"] = pd.Series([None] * len(official_results), dtype="object")
    official_results["probability"] = np.nan
    model.eval()
    with torch.no_grad():
        for start in tqdm(range(0, len(official_results), batch_size), desc="Official predictions"):
            batch = official_results.iloc[start:start + batch_size]
            batch = batch[batch["image_status"] == "ok"]
            if len(batch) == 0:
                continue
            images = torch.stack([validation_transform(load_image(name)) for name in batch["filename"]]).to(device)
            probabilities = model(images).softmax(dim=1)
            values, numbers = probabilities.max(dim=1)
            official_results.loc[batch.index, "predicted_label"] = [class_names[number] for number in numbers.cpu().tolist()]
            official_results.loc[batch.index, "probability"] = values.cpu().numpy()
    official_results.to_csv(run_name + "_official_predictions.csv", index=False)
    print("Official rows saved:", len(official_results))

## 17. Optional: predict one image

After loading the best checkpoint, enter an image filename to see the five highest-scoring known classes.


In [ ]:
new_image_filename = ""
if new_image_filename:
    check_checkpoint_preprocessing(checkpoint)
    image_path = Path(new_image_filename).expanduser()
    if image_path.is_file():
        image_path = image_path.resolve()
    image = validation_transform(load_image(image_path)).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        probabilities = model(image).softmax(dim=1)[0]
    values, numbers = probabilities.topk(min(5, len(class_names)))
    print(pd.DataFrame({"diagnosis": [class_names[number] for number in numbers.cpu().tolist()],
                        "probability": values.cpu().tolist()}))